Customer Analysis
Ranks customers by sales, profit, and volume contribution, with precomputed percentile ranks for instant threshold filtering. Excludes generic/non-identifiable cash-sale accounts (resolved upstream in clean_to_silver_analysis).

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.analysis.customer_analysis import build_customer_summary, filter_by_percentile, get_lapsed_top_customers
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"

Load analysis silver

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])

print(f"Analysis silver: {analysis_silver.shape}")

Build customer summary

In [0]:
customer_summary = build_customer_summary(analysis_silver)
print(customer_summary.shape)
print(customer_summary[["customer_name", "total_sales", "sales_percentile", "total_profit", "profit_percentile"]].head(10))

Save

In [0]:
save_gold(blob_service, customer_summary, f"{ANALYSIS_BASE}/customer_ranking.parquet")

buffer = io.BytesIO()
customer_summary.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob= f"{ANALYSIS_BASE}/customer_ranking.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved customer_ranking.parquet and .xlsx")

Demo: threshold as pure filter

In [0]:
dbutils.widgets.text("sales_percentile_threshold", "90")
sales_threshold = float(dbutils.widgets.get("sales_percentile_threshold"))

top_customers = filter_by_percentile(customer_summary, "sales", sales_threshold)
print(f"Customers at or above the {sales_threshold}th percentile: {len(top_customers)}")
print(top_customers[[
    "customer_name", "customer_address", "customer_phone1", "customer_email",
    "total_sales", "total_profit", "total_units", "order_count", "last_purchase"
]].head(30))

Lapsed top customers (re-engagement candidates)

In [0]:
lapsed_top_customers = get_lapsed_top_customers(customer_summary, "sales", 90)

lapsed_businesses = lapsed_top_customers[~lapsed_top_customers["is_individual"]].sort_values("total_sales", ascending=False)
lapsed_individuals = lapsed_top_customers[lapsed_top_customers["is_individual"]]

print("Lapsed top customers — businesses (real re-engagement targets):")
print(f"Count: {len(lapsed_businesses)}")
print(lapsed_businesses[["customer_name", "total_sales", "last_purchase", "days_since_last_purchase"]].head(20))

print("\nLapsed top customers — individuals (likely just haven't needed a replacement yet):")
print(f"Count: {len(lapsed_individuals)}")